In [1]:
import pennylane as qml
import numpy as np
import torch

/opt/miniconda3/lib/python3.12/site-packages/pennylane/__init__.py:212: PennyLaneDeprecationWarning: PennyLane v0.44 has dropped maintainence support for NumPy < 2.0.0. You have version 1.26.4 installed. Future versions of PennyLane will not work with NumPy<2.0. Please consider upgrading NumPy using `python -m pip install numpy --upgrade`. 
  warnings.warn(
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x10d3437d0>>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [2]:
seed = 4321
np.random.seed(seed=seed)
torch.manual_seed(seed=seed)

In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

X,y = load_breast_cancer(return_X_y=True)
x_tr, x_test, y_tr, y_test = train_test_split(X, y, train_size=0.8, shuffle=True)
x_val, x_test, y_val, y_test = train_test_split(x_test, y_test, train_size=0.5, shuffle=True)

In [4]:
print(len(x_tr))

455


La base de datos que vamos a emplear tiene 30 variables, y a dia de hoy no tenemos acceso a ordenadores con 30 qubits, por lo que hemos de considerar una de las siguientes opciones: 
1. Usar amplitude encoding como feature map sobre 5 qubits (2**5 = 32)
2. Usar cualquier otro feature map pero con una reduccion de la dimensionalidad

METODO 1: AMPLITUDE ENCODING

METODO 2: REDUCCION DE LA DIMENSIONALIDAD

Vamos a restringir el caso a un circuito de 4 qubits

In [5]:
pca = PCA(n_components=4)
xs_tr = pca.fit_transform(x_tr)
xs_test = pca.transform(x_test)
xs_val = pca.transform(x_val)

xs_tr = torch.FloatTensor(xs_tr)
xs_test = torch.FloatTensor(xs_test)
xs_val = torch.FloatTensor(xs_val)

y_tr = torch.FloatTensor(y_tr)
y_test = torch.FloatTensor(y_test)
y_val = torch.FloatTensor(y_val)

reps = 4
nqubits = 4
theta = torch.randn((reps+1, nqubits), requires_grad=True)  # 9x4

In [6]:
print(len(xs_tr[0]))

4


In [7]:
# Vamos a usar ZZ feature map y la forma variacional two-local
from itertools import combinations

def ZZFeatureMap(nqubits, data):


    nload = min(len(data), nqubits)
    for i in range(nload):
        qml.Hadamard(i)
        qml.RZ(2 * data[i], wires = i)
    
    for pair in list(combinations(range(nload),2)):
        q0 = pair[0]
        q1 = pair[1]

        qml.CNOT(wires=[q0, q1])
        qml.RZ(2.0 * (np.pi-data[q1])*(np.pi-data[q0]), wires=q1)
        qml.CNOT(wires=[q0, q1])

def TwoLocal(nqubits, theta, reps):
    for r in range(reps):
        for j in range(nqubits):
            qml.RY(theta[r,j], wires=j)
        
        for j in range(nqubits-1):
            qml.CNOT(wires=[j,j+1])
    
    for j in range(nqubits):
        qml.RY(theta[reps,j], wires=j)


En lugar de pedir a pennylane que devuelva las probabilidades de medida, le pediremos que devuelva el valor esperado del operador:
$M =
\begin{pmatrix}
1 & 0\\
0 & 0
\end{pmatrix}$

In [8]:
state_0 = np.array([[1],[0]])
M = state_0 @ state_0.T.conj()

In [9]:
nqubits = 4
dev = qml.device('default.qubit', wires=nqubits)

def qnn_circuit(inputs, theta):
    ZZFeatureMap(nqubits=nqubits, data=inputs)
    TwoLocal(nqubits=nqubits, theta=theta, reps=reps) # por simplicidad se pide una repeticion unicamente
    return(qml.expval(qml.Hermitian(M, wires=0)))

# se añade el argumento interface = 'torch' al inicializar el nodo para que trabaje con pytorch
qnn = qml.QNode(qnn_circuit, dev, interface='torch')

In [10]:
# Por la interoperabilidad de pennylane podremos usar pytorch para entrenar la red neuronal cuantica
# creamos una capa torch que contenga nuestra red neuronal cuantica

# Dado que en la forma variacional empleamos 1 repeticion y hay 4 qubits
# necesitamos 8 parametros entrenables, se indican en weight_shapes con el
# nombre de esos parametros
weight_shapes = {'theta': (reps+1,4)}
qlayer = qml.qnn.TorchLayer(qnode=qnn, weight_shapes=weight_shapes)

In [11]:
# estamos creado un circuito cuantico pero con pytorch podremos hallar 
# su gradiente para poder entrenarlo

# incluimos una capa clasica antes del circuito que incluye 2 neuronas
# y otra tras el circuito que incluye 2 neuronas tambien
clayer_1 = torch.nn.Linear(2,2) # input node
clayer_2 = torch.nn.Linear(2,2) # output mode
layers = [clayer_1, qlayer, clayer_2]
model = torch.nn.Sequential(*layers)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
loss = torch.nn.CrossEntropyLoss()
criterion = torch.nn.CrossEntropyLoss()

In [16]:
theta = torch.randn((2, nqubits), requires_grad=True)

optimizer = torch.optim.Adam([theta], lr=0.01)
loss_fn = torch.nn.CrossEntropyLoss()
epochs = 50
outputs = []
losses = []
def model(x, theta):
    return qnn(x, theta)

for epoch in range(epochs):
    optimizer.zero_grad()
    
    outputs = []
    for x in xs_tr:
        outputs.append(qlayer(x))  # solo pasas el input
    
    outputs = torch.stack(outputs).view(-1)
    loss = loss_fn(outputs, y_tr)
    losses.append(loss.detach().numpy())

    if epoch % 10 == 0:
        print(f'Epoch: {epoch} and loss: {loss}')
    
    loss.backward()
    optimizer.step()

Epoch: 0 and loss: 1751.833740234375
Epoch: 10 and loss: 1751.833740234375
Epoch: 20 and loss: 1751.833740234375
Epoch: 30 and loss: 1751.833740234375
Epoch: 40 and loss: 1751.833740234375


In [17]:
def predict(model, X, threshold=0.5):
    preds = []
    for x in X:
        out = model(x)
        preds.append((out > threshold).float())
    return torch.stack(preds).squeeze()

In [22]:
y_pred = predict(qlayer, xs_test)
for item in y_pred:
    if item >= 0.5:
        item = 1.0
    else:
        item = 0.0

accuracy = (y_pred == y_test).float().mean()
print("Accuracy:", accuracy.item())

Accuracy: 0.4912280738353729
